# Cierre: síntesis y tarjeta de cada modelo

## Qué responde este notebook

Resume lo que respondió cada etapa, con el notebook donde está el detalle, y arma una tarjeta por
modelo a partir de la metadata que guardó
[`6.1_modelo_final_y_temporada_actual.ipynb`](6.1_modelo_final_y_temporada_actual.ipynb). Las cifras
de la tarjeta se leen de esos archivos, no se copian a mano. El índice completo de notebooks está en
[`README.md`](README.md).

## 1. Datos y variables (etapas 1 a 3)

- **Fuente**: `nflreadpy`, auditada por fuente en
  [`1.1_calidad_fuentes.ipynb`](1.1_calidad_fuentes.ipynb) y cargada con `weekly/wr/src/data.py`
  ([`1.2_construccion_datos.ipynb`](1.2_construccion_datos.ipynb)). Las alineaciones de 2025 en
  adelante usan otro esquema; se unificaron con las históricas en
  [`1.3_unificacion_alineaciones.ipynb`](1.3_unificacion_alineaciones.ipynb). Las históricas,
  además, usaban códigos de equipo de su época (`OAK`, `SD`); se traducen a los actuales en `data.py`
  (ver [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)).
- **Sin fuga**: cada promedio de un receptor usa solo partidos anteriores
  ([`1.4_promedios_sin_fuga.ipynb`](1.4_promedios_sin_fuga.ipynb)).
- **Lo que mostró el análisis exploratorio** (2.1 a 2.6): recepciones y yardas dependen sobre todo de
  la participación del receptor en su ofensiva; los touchdowns son un evento raro, con la mayoría de
  los partidos en cero.
- **Variables** ([`3.1_ingenieria_features.ipynb`](3.1_ingenieria_features.ipynb) y
  [`3.2_seleccion_features.ipynb`](3.2_seleccion_features.ipynb)): 34 variables para los modelos de
  árboles y 30 para los lineales, descritas en el
  [glosario](../../../docs/data/GLOSARIO_VARIABLES.md).

## 2. Cómo se mide y cómo se elige

Los modelos predicen valores esperados, que se suman para armar puntos de fantasy. Por eso la métrica
que decide tiene que premiar la media
([`4.1_preparacion_y_metricas.ipynb`](4.1_preparacion_y_metricas.ipynb),
[`docs/decisions/0004-metricas-de-seleccion.md`](../../../docs/decisions/0004-metricas-de-seleccion.md)):

| Resultado | Métrica que decide (validación 2022-2023) | Requisitos para ser elegible |
|---|---|---|
| `receptions`, `receiving_yards` | RMSE | R² fuera de muestra > 0 en validación y en cada año; sesgo adicional ≤ 5%; pendiente de calibración 0.9-1.1 |
| `receiving_tds` | Deviance de Poisson | Los mismos, más D² fuera de muestra > 0 |

El MAE premia la mediana, y en touchdowns la mediana es cero. La prueba (2024-2025) solo se reporta,
y una diferencia entre dos modelos cuenta solo si su intervalo bootstrap por semanas no incluye cero,
ajustado por Bonferroni cuando se comparan varias alternativas a la vez.

## 3. El modelo elegido de cada resultado

- **Recepciones y yardas**: XGBoost con error cuadrático
  ([`4.2_modelos_recepciones_yardas.ipynb`](4.2_modelos_recepciones_yardas.ipynb)). En yardas,
  RandomForest quedó apenas abajo en RMSE de validación, con un intervalo que incluye cero; por la
  regla de empates se queda XGBoost.
- **Touchdowns**: XGBoost con objetivo Poisson
  ([`4.3_modelos_touchdowns.ipynb`](4.3_modelos_touchdowns.ipynb)). El modelo anterior (Tweedie con
  hiperparámetros por defecto, elegido por MAE) tenía más descalibración que discriminación y por
  eso su R² salía negativo; el elegido tiene R² fuera de muestra de 0.08 en validación.
- **Ajuste de hiperparámetros**
  ([`4.4_ajuste_hiperparametros.ipynb`](4.4_ajuste_hiperparametros.ipynb)): una búsqueda compacta no
  mejora ninguno de los tres de forma real, así que se queda la configuración de partida: profundidad
  3 en los tres; 200 árboles con tasa 0.03 en recepciones y yardas, y 200 árboles con tasa 0.08 en
  touchdowns.

## 4. Alternativas probadas que no se adoptan

Cada alternativa se comparó contra el modelo elegido con la misma regla. Ninguna se adopta:

| Notebook | Alternativa | Resultado |
|---|---|---|
| [`4.5`](4.5_regresion_cuantiles.ipynb) | Rango P10/P50/P90 con regresión de cuantiles | Útil como rango en recepciones y yardas (cobertura de prueba cercana al 80%); el P50 es la mediana: gana en MAE y pierde en RMSE y R² |
| [`4.6`](4.6_ensamble_simple.ipynb) | Promedios de modelos; mezcla con el modelo de dos etapas en touchdowns | Empates, también la mejor mezcla en touchdowns (70/30) |
| [`4.7`](4.7_jerarquico_touchdowns.ipynb) | Jerárquico con Binomial Negativa en touchdowns | Peor (+0.0129 de deviance) y descalibrado (pendiente 0.80); mezclado con el XGBoost (90/10), empate |
| [`4.8`](4.8_jerarquico_recepciones_yardas.ipynb) | Jerárquico en recepciones y yardas | Peor en los dos (+0.121 y +3.47 de RMSE); en yardas, la escala logarítmica predice cerca de la mediana |
| [`4.9`](4.9_ventana_entrenamiento.ipynb) | Ventanas más cortas, más peso a lo reciente, tendencia por jugador | Empates o peor, ya con el intervalo de 95% |

De estas pruebas, `weekly/wr/models/exploratorio/` guarda solo los modelos de cuantiles de 4.5 y el
jerárquico de 4.7, que reutiliza 4.9 y no se versiona, más la metadata de 4.6 y 4.7. El flujo semanal
no usa ninguno.

## 5. Evaluación

- **Estable año con año** ([`5.1_estabilidad_temporal.ipynb`](5.1_estabilidad_temporal.ipynb)): en
  los cinco cortes de 2021 a 2025, R² fuera de muestra de 0.43-0.47 en recepciones, 0.35-0.39 en
  yardas y 0.079-0.089 en touchdowns; los tres cumplen los requisitos en cada año.
- **Mejor que el flujo heredado** ([`5.2_benchmark_vs_legado.ipynb`](5.2_benchmark_vs_legado.ipynb)):
  sobre las mismas semanas 7-18 de 2025, mejor en yardas y touchdowns con diferencias que no incluyen
  cero aun ajustando por las tres comparaciones; en recepciones, mejor en la estimación puntual pero
  empate con el intervalo ajustado. Mejor en MAE en los tres, la métrica del flujo heredado.
- **Explicable** ([`5.3_explicabilidad_y_casos.ipynb`](5.3_explicabilidad_y_casos.ipynb)): tres
  métodos de importancia coinciden en participación reciente y lugar en la alineación; el nivel de
  carrera pesa en SHAP y en la importancia nativa, no en la permutación de 3.2. El modelo acierta en
  los partidos típicos de cada perfil y se queda corto en las semanas atípicas; por cuartil de
  participación previa queda a menos de 2 yardas de lo real.
- **Temporada en curso**
  ([`6.1_modelo_final_y_temporada_actual.ipynb`](6.1_modelo_final_y_temporada_actual.ipynb)): en las
  semanas 1-4 de 2026, que no se usaron en ningún paso, los tres quedan al nivel de validación en R²
  fuera de muestra. En la métrica principal, yardas y touchdowns quedan un poco por encima de
  validación y prueba, y el modelo subestima (−3.1 yardas por receptor).

## 6. Tarjeta de cada modelo

Leída de `weekly/wr/models/{target}_metadata.json`, la misma metadata que usa el flujo semanal.

In [1]:
import json
from pathlib import Path

import pandas as pd

pd.set_option("display.width", 160)
pd.set_option("display.max_colwidth", 120)

TARGETS = ("receptions", "receiving_yards", "receiving_tds")
metadata = {t: json.loads(Path(f"../models/{t}_metadata.json").read_text()) for t in TARGETS}

tarjeta = {}
for target, m in metadata.items():
    principal = m["metrica_principal"]
    referencia = m["metricas_referencia"]
    h = m["hiperparametros"]
    tarjeta[target] = {
        "modelo": f"XGBoost {h.get('objective')}, profundidad {h.get('max_depth')}, {h.get('n_estimators')} árboles, tasa {h.get('learning_rate')}",
        "entrenado con": m["entrenado_con"].split(";")[0],
        "variables": len(m["features_arbol_en_orden"]),
        "media de entrenamiento": round(m["media_entrenamiento"], 4),
        "métrica principal": principal,
        "validación 2022-2023": referencia["validacion"][principal],
        "prueba 2024-2025": referencia["prueba"][principal],
        f"2026 (semanas {referencia['temporada_2026']['semanas']})": referencia["temporada_2026"][principal],
        "R² fuera de muestra: validación / prueba / 2026": " / ".join(
            f"{referencia[p]['r2_oos']:.3f}" for p in ("validacion", "prueba", "temporada_2026")),
        "R² fuera de muestra mínimo por año (2021-2025)": min(a["r2_oos"] for a in referencia["walk_forward"].values()),
        "pendiente de calibración: validación / 2026": " / ".join(
            f"{referencia[p]['pendiente_calibracion']:.3f}" for p in ("validacion", "temporada_2026")),
        "cumple requisitos": referencia["cumple_requisitos"],
    }
pd.DataFrame(tarjeta)

,receptions,receiving_yards,receiving_tds
modelo,"XGBoost reg:squarederror, profundidad 3, 200 árboles, tasa 0.03","XGBoost reg:squarederror, profundidad 3, 200 árboles, tasa 0.03","XGBoost count:poisson, profundidad 3, 200 árboles, tasa 0.08"
entrenado con,2016-2025 (temporadas completas),2016-2025 (temporadas completas),2016-2025 (temporadas completas)
variables,34,34,34
media de entrenamiento,2.6981,34.2784,0.2088
métrica principal,rmse,rmse,deviance_poisson
validación 2022-2023,1.9293,29.5718,0.6167
prueba 2024-2025,1.8258,27.9143,0.6252
2026 (semanas 1-4),1.8597,29.6554,0.6302
R² fuera de muestra: validación / prueba / 2026,0.438 / 0.459 / 0.462,0.362 / 0.375 / 0.361,0.083 / 0.088 / 0.083
R² fuera de muestra mínimo por año (2021-2025),0.4305,0.3521,0.0792


In [2]:
for target, m in metadata.items():
    print(f"{target}, limitaciones conocidas:")
    for limitacion in m["limitaciones_conocidas"]:
        print(f"  - {limitacion}")
print("\nrequisitos:", metadata["receptions"]["requisitos"])
print("\nreferencias de la politica de metricas:")
for referencia in metadata["receptions"]["referencias"]:
    print(f"  - {referencia}")

receptions, limitaciones conocidas:
  - predice valores esperados: queda por debajo en semanas de produccion extrema y por encima en semanas sin produccion; no anticipa semanas atipicas (ver 5.3_explicabilidad_y_casos.ipynb)
  - depth_team tiene dato en 95.4% de las filas de 2026 ya jugadas; los faltantes son receptores que no aparecen en la alineacion de esa semana (ver 1.3_unificacion_alineaciones.ipynb)
receiving_yards, limitaciones conocidas:
  - predice valores esperados: queda por debajo en semanas de produccion extrema y por encima en semanas sin produccion; no anticipa semanas atipicas (ver 5.3_explicabilidad_y_casos.ipynb)
  - depth_team tiene dato en 95.4% de las filas de 2026 ya jugadas; los faltantes son receptores que no aparecen en la alineacion de esa semana (ver 1.3_unificacion_alineaciones.ipynb)
receiving_tds, limitaciones conocidas:
  - predice valores esperados: queda por debajo en semanas de produccion extrema y por encima en semanas sin produccion; no anticipa sem

## 7. Limitaciones y pendientes

- **Selección de variables**: `3.2_seleccion_features.ipynb` midió la importancia con MAE y usó
  2024-2025 como validación, los mismos años que después son prueba (ver
  [`docs/HALLAZGOS.md`](../../../docs/HALLAZGOS.md)). Las verificaciones que no comparten ese uso son
  el corte de 2021 del walk-forward y la temporada 2026. Además, con el dato de alineaciones
  corregido, su regla daría 31 variables en lugar de las 34 que usan los modelos: queda pendiente
  rehacerla con un método estable.
- **Touchdowns**: el R² alcanzable es bajo porque la mayor parte de la variación entre partidos es
  azar; el modelo supera al promedio histórico, pero por poco.
- **A vigilar en el flujo semanal**: la subestimación de las primeras semanas de 2026 y la pendiente
  de calibración de touchdowns, la que más varía entre años (de 0.947 a 1.095).
- **Fuera del flujo semanal**: el rango P10/P50/P90
  ([`4.5_regresion_cuantiles.ipynb`](4.5_regresion_cuantiles.ipynb)) y los modelos exploratorios de
  `weekly/wr/models/exploratorio/`.
- **Siguientes fases** ([`docs/PLAN.md`](../../../docs/PLAN.md)): registro de cada corrida semanal
  (`tracking.csv`) y una guía para adaptar este flujo a las demás posiciones.

Anterior: [`6.1_modelo_final_y_temporada_actual.ipynb`](6.1_modelo_final_y_temporada_actual.ipynb)